# ========================================
# Ассистент анализа эмоций
# ========================================

In [ ]:
from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.tools import Tool, ToolParameter, ToolRegistry
from typing import Dict, Any, List
from paddlenlp import Taskflow
import ast
import os
import pandas as pd
import re


In [ ]:

os.environ["LLM_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["LLM_API_KEY"] = ""  # Ваш ключ
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1"
os.environ["LLM_TIMEOUT"] = "60"


# ========================================
# 1. Определите инструменты анализа кода# ========================================

очистка текста

In [ ]:
class ProcessChatHistoryTool(Tool):
    """
    Импорт и очистка чата WeChat/QQ
Унаследовать абстрактный класс Tool, реализовать run, get_parameters метод
    """
    def __init__(self):
        super().__init__(
            name="process_chat_history",
            description="Чтение TXT чата WeChat/QQ, очистка, DataFrame"
        )

    def run(self, parameters: Dict[str, Any]) -> pd.DataFrame:
        """
        Точка входа инструмента
        :param parameters: file_path, chat_type
        :return: очищенный DataFrame
        """
        # Получение параметров
        file_path = parameters.get("file_path", "")
        chat_type = parameters.get("chat_type", "wechat")

        messages = []
        pattern = re.compile(r'(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\s+(.+?):\s+(.+)')

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    match = pattern.match(line)
                    if match:
                        time, sender, content = match.groups()

                        # Фильтр системных сообщений
                        if any(keyword in content for keyword in ['[图片]', '[视频]', '撤回了一条消息', '拍了拍']):
                            continue

                        messages.append({
                            'time': time,
                            'sender': sender,
                            'content': content
                        })

            df = pd.DataFrame(messages)
            print(f"✅Успешно импортировано{len(df)} Действительная запись чата!")
            return df

        except Exception as e:
            print(f"❌ Ошибка чтения: {str(e)}")
            return pd.DataFrame()

    def get_parameters(self) -> List[ToolParameter]:
        """
        Параметры инструмента
        """
        return [
            ToolParameter(
                name="file_path",
                type="string",
                description="Путь к TXT файлу чата",
                required=True
            ),
            ToolParameter(
                name="chat_type",
                type="string",
                description="Тип: wechat или qq",
                required=False
            )
        ]

анализ настроений

In [ ]:
class AnalyzeSentimentAndMoodTool(Tool):
    """ИспользованиеSKEP-Модель ERNIE анализирует эмоции и настроения записей чата"""
    
    def __init__(self):
        super().__init__(
            name="analyze_sentiment_and_mood",
            description="Тональность (позитив/негатив) и настроение в чате"
        )
        # Инициализация модели (один раз)
        self.sentiment_analyzer = Taskflow(
            "sentiment_analysis", 
            model="skep_ernie_1.0_large_ch",
        )

    def run(self, parameters: Dict[str, Any]) -> pd.DataFrame:
        df = parameters.get("df", pd.DataFrame())
        
        if df.empty:
            return df

        contents = df['content'].tolist()

        try:
            results = self.sentiment_analyzer(contents)

            sentiments = [res['sentiment_key'] for res in results]
            confidence = [
                res['positive_probs'] if res['sentiment_key'] == 'positive' 
                else 1 - res['positive_probs'] 
                for res in results
            ]

            moods = []
            for res in results:
                if res['sentiment_key'] == 'positive':
                    moods.append('радость/одобрение')
                else:
                    neg_prob = 1 - res['positive_probs']
                    if neg_prob > 0.8:
                        moods.append('злость/грусть')
                    else:
                        moods.append('безразличие/нейтрально')

            df['sentiment'] = sentiments
            df['mood'] = moods
            df['confidence'] = confidence

            print("✅ Анализ эмоций и настроения завершен!")
            return df

        except Exception as e:
            print(f"❌ Ошибка анализа настроений:{e}")
            return df

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="df",
                type="object",
                description="Очищенный DataFrame чата",
                required=True
            )
        ]

статистика настроений

In [ ]:
class SummarizeEmotionStatsTool(Tool):
    """Статистика передает эмоциональные данные, генерирует отчеты и структурированные результаты."""
    
    def __init__(self):
        super().__init__(
            name="summarize_emotion_stats",
            description="Подсчёт эмоций, доли, словарь отчёта"
        )

    def run(self, parameters: Dict[str, Any]) -> dict:
        df = parameters.get("df", pd.DataFrame())
        sender_name = parameters.get("sender_name", None)
        
        if df.empty or 'sentiment' not in df.columns:
            print("❌ Нет данных — сначала запустите предыдущие инструменты!")
            return {}

        if sender_name:
            analysis_df = df[df['sender'] == sender_name].copy()
            if analysis_df.empty:
                print(f"⚠️ не найдено {sender_name} история чата")
                return {}
            print(f"🔍 Статистика для {sender_name}...")
        else:
            analysis_df = df.copy()
            print("🔍 Статистика для всех...")

        total_messages = len(analysis_df)
        happy_count = len(analysis_df[analysis_df['sentiment'] == 'positive'])
        angry_count = len(analysis_df[analysis_df['sentiment'] == 'negative'])

        happy_ratio = round((happy_count / total_messages) * 100, 2) if total_messages > 0 else 0.0
        angry_ratio = round((angry_count / total_messages) * 100, 2) if total_messages > 0 else 0.0

        print("\n" + "="*30)
        print(f"📊 【Отчет по статистике эмоций】")
        print(f"Всего сообщений: {total_messages}")
        print(f"😄 Радость: {happy_count} ({happy_ratio}%)")
        print(f"😡 злость/грусть: {angry_count} сообщ. (Пропорция {angry_ratio}%)")
        print(f"😐 нейтральный/другой: {total_messages - happy_count - angry_count} сообщ.")
        print("="*30 + "\n")

        return {
            'total_messages': total_messages,
            'happy_count': happy_count,
            'angry_count': angry_count,
            'happy_ratio': happy_ratio,
            'angry_ratio': angry_ratio
        }

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="df",
                type="object",
                description="DataFrame с анализом эмоций",
                required=True
            ),
            ToolParameter(
                name="sender_name",
                type="string",
                description="Опционально: имя отправителя",
                required=False
            )
        ]

In [ ]:
class PlotEmotionChartTool(Tool):
    """Столбчатая диаграмма эмоций"""
    
    def __init__(self):
        super().__init__(
            name="plot_emotion_chart",
            description="Визуализация словаря статистики"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        stats = parameters.get("stats", {})
        
        if not stats:
            return "⚠️ Нет данных для графика"

        # Настройка шрифта для кириллицы
        plt.rcParams['font.sans-serif'] = ['SimHei']
        plt.rcParams['axes.unicode_minus'] = False

        labels = ['радость', 'злость/грусть']
        counts = [stats['happy_count'], stats['angry_count']]
        colors = ['#FF9999', '#66B2FF']

        plt.figure(figsize=(8, 5))
        bars = plt.bar(labels, counts, color=colors)
        plt.title(f"Распределение эмоций (всего: {stats['total_messages']})", fontsize=15)
        plt.ylabel('Сообщений', fontsize=12)

        # Подписи значений
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, int(yval), ha='center', va='bottom', fontsize=12)

        plt.show()
        return "✅ График построен!"

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="stats",
                type="object",
                description="Словарь из summarize_emotion_stats",
                required=True
            )
        ]

# ========================================
# 2. Создание реестров инструментов и агентов
# ========================================

In [ ]:
tool_registry = ToolRegistry()

tool_registry.register_tool(ProcessChatHistoryTool())
tool_registry.register_tool(AnalyzeSentimentAndMoodTool())
tool_registry.register_tool(SummarizeEmotionStatsTool())
tool_registry.register_tool(PlotEmotionChartTool())

print("✅ Все инструменты зарегистрированы!")

# ========================================
# 3. Инициализируем большую модель# ========================================

In [ ]:
print(">>> Base URL:", repr(os.getenv("LLM_BASE_URL")))
	llm = HelloAgentsLLM(
    model="Qwen/Qwen3-8B",
    base_url="https://api-inference.modelscope.cn/v1",
    api_key="YOUR API KEY",
    timeout=60
)

# ========================================
# 4. Определение слов системных подсказок
# ========================================

In [ ]:
system_prompt = """Ты психолог отношений с 10-летним опытом и коуч по коммуникации. Проанализируй чат и дай проницательный отчёт.

Выполни шаги:
1. **Контекст**: стадия отношений (флирт, влюблённость, конфликт).
2. **Подтекст**: скрытые эмоции и невысказанные потребности.
3. **Количественная оценка**: индекс влечения 0–100.
4. **Советы по ответам**: 3 варианта (юмор, искренность, лёгкий флирт).

Отчёт в Markdown:
- **Индекс влечения**: (балл и комментарий)
- **Глубокий разбор**: (психология и намерения)
- **Перевод подтекста**: (1–2 ключевые реплики)
- **Варианты ответов**: (3 конкретных варианта)
"""

# ========================================
# 5. Генерация агента# ========================================

In [ ]:
agent = SimpleAgent(
name="Ассистент анализа эмоций",
llm=llm,
system_prompt=system_prompt,
tool_registry=tool_registry
)

# ========================================
# 6. Запустите пример
# ========================================

In [ ]:
with open("data/1.txt","r",encoding="utf-8") as f:
  talktxt=f.read()

print('---------------Чат---------------')
print(talktxt)

print('--------------Анализ--------------')
print("текущий LLM_BASE_URL:", repr(os.environ["LLM_BASE_URL"]))
result=agent.run(talktxt)
print(result)
print('---------------Сохранить результаты---------------')
with open("outputs/review_report.md", "w", encoding="utf-8") as f:
  f.write(result)
print("\nОтчёт сохранён в outputs/review_report.md")